In [15]:
import jax.numpy as jnp

from shapiq.game.table import TableGame
from shapiq.players import FinitePlayers

In [16]:
coalitions = jnp.array(
    [
        [0, 0, 0],  # empty
        [1, 0, 0],  # {0}
        [0, 1, 0],  # {1}
        [1, 1, 0],  # {0, 1}
        [0, 0, 1],  # {2}
        [1, 0, 1],  # {0, 2}
        [0, 1, 1],  # {1, 2}
        [1, 1, 1],  # {0, 1, 2}
    ],
    dtype=bool,
)
g = TableGame(jnp.array([[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]), FinitePlayers(3))
print("game batch:", g.shape, "shared players:", g.players.shape)
for coalition, value in zip(coalitions.tolist(), g(coalitions).tolist(), strict=True):
    print(coalition, "->", value)

game batch: (1,) shared players: ()
[False, False, False] -> 0.0
[True, False, False] -> 1.0
[False, True, False] -> 2.0
[True, True, False] -> 4.0
[False, False, True] -> 3.0
[True, False, True] -> 5.0
[False, True, True] -> 6.0
[True, True, True] -> 10.0


In [17]:
from shapiq.game.exact import ExactComputer
from shapiq.index import ShapleyInteractionIndex

computer = ExactComputer.from_game(g)

shapley_index = ShapleyInteractionIndex(max_order=2)
shapley_interactions = computer(shapley_index)
shapley_interactions

SetMoebiusGame(values=Array([[2.333334 , 3.3333342, 1.5000007, 4.3333344, 1.5000007, 1.500001 ]],      dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

In [18]:
import jax
import jax.numpy as jnp

from shapiq.sampling._base import PairedSizeWithoutReplacement

sampler = PairedSizeWithoutReplacement(
    keys=jax.random.key(
        1
    ),  # TODO(advueu963): Keys and Weights should broadcast together, and not keys dictate the batch size.  # noqa: TD003
    weights=jnp.ones(g.n_players + 1),
)
sampler

PairedSizeWithoutReplacement(keys=Array((), dtype=key<fry>) overlaying:
[0 1], cursor=Array(0, dtype=int32), weights=Array([1., 1., 1., 1.], dtype=float32))

In [19]:
from shapiq.approximator.base import SHAPIQ
from shapiq.index import ShapleyInteractionIndex

index = ShapleyInteractionIndex(max_order=2)
approximator = SHAPIQ(sampler=sampler, game=g, index=index)
estimate, approximator = approximator(4)

In [20]:
estimate

SetMoebiusGame(values=Array([[ 4.833335 ,  0.3333335, -5.500003 ,  4.833335 , 15.500008 ,
        -5.500003 ]], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

In [21]:
shapley_interactions

SetMoebiusGame(values=Array([[2.333334 , 3.3333342, 1.5000007, 4.3333344, 1.5000007, 1.500001 ]],      dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

In [22]:
approximator

SHAPIQ(sampler=PairedSizeWithoutReplacement(keys=Array((), dtype=key<fry>) overlaying:
[0 1], cursor=Array(2, dtype=int32), weights=Array([1., 1., 1., 1.], dtype=float32)), game=TableGame(values=Array([[ 0.,  1.,  2.,  4.,  3.,  5.,  6., 10.]], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3)), index=ShapleyInteractionIndex(min_order=1, max_order=2, alpha=Array(1., dtype=float32), beta=Array(1., dtype=float32)), history=DrawHistory(samplers=PairedSizeWithoutReplacement(keys=Array((1,), dtype=key<fry>) overlaying:
[[0 1]], cursor=Array([0], dtype=int32), weights=Array([[1., 1., 1., 1.]], dtype=float32)), coalitions=Array([[False, False, False],
       [ True,  True,  True],
       [False,  True, False],
       [ True, False,  True]], dtype=bool), values=Array([[ 0., 10.,  2.,  5.]], dtype=float32), ends=Array([4], dtype=int32)))

## Batched Sampler && Batched GAMES

In [23]:
import jax
import jax.numpy as jnp

from shapiq.sampling._base import PairedSizeWithoutReplacement

sampler = PairedSizeWithoutReplacement(
    keys=jax.random.split(jax.random.key(1), 2),
    weights=jnp.array([jnp.ones(g.n_players + 1), jnp.array([1.0, 2.0, 3.0, 4.0])]),
)
sampler

PairedSizeWithoutReplacement(keys=Array((2,), dtype=key<fry>) overlaying:
[[ 507451445 1853169794]
 [1948878966 4237131848]], cursor=Array(0, dtype=int32), weights=Array([[1., 1., 1., 1.],
       [1., 2., 3., 4.]], dtype=float32))

In [24]:
from shapiq.approximator.base import SHAPIQ
from shapiq.index import ShapleyInteractionIndex

g = TableGame(
    jnp.array(
        [[0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0], [0.0, 1.0, 2.0, 4.0, 3.0, 5.0, 6.0, 10.0]]
    ),
    FinitePlayers(3),
)

index = ShapleyInteractionIndex(max_order=2)
approximator = SHAPIQ(sampler=sampler, game=g, index=index)
estimate, approximator = approximator(4)
estimate.shape

(2,)

In [25]:
estimate, approximator = approximator(4)

In [26]:
estimate[0]

SetMoebiusGame(values=Array([2.333334 , 3.3333342, 1.5000005, 4.333335 , 1.5000005, 1.5000007],      dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

In [27]:
shapley_interactions

SetMoebiusGame(values=Array([[2.333334 , 3.3333342, 1.5000007, 4.3333344, 1.5000007, 1.500001 ]],      dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3), interactions=Array([[ True, False, False],
       [False,  True, False],
       [ True,  True, False],
       [False, False,  True],
       [ True, False,  True],
       [False,  True,  True]], dtype=bool), additivity=2)

In [28]:
approximator[1]

SHAPIQ(sampler=PairedSizeWithoutReplacement(keys=Array((), dtype=key<fry>) overlaying:
[1948878966 4237131848], cursor=Array(4, dtype=int32), weights=Array([1., 2., 3., 4.], dtype=float32)), game=TableGame(values=Array([ 0.,  1.,  2.,  4.,  3.,  5.,  6., 10.], dtype=float32), players=FinitePlayers(ids=Array([0, 1, 2], dtype=int32), players=None, latent_n_players=3, n_players=3)), index=ShapleyInteractionIndex(min_order=1, max_order=2, alpha=Array(1., dtype=float32), beta=Array(1., dtype=float32)), history=DrawHistory(samplers=PairedSizeWithoutReplacement(keys=Array((2,), dtype=key<fry>) overlaying:
[[1948878966 4237131848]
 [1948878966 4237131848]], cursor=Array([0, 2], dtype=int32), weights=Array([[1., 2., 3., 4.],
       [1., 2., 3., 4.]], dtype=float32)), coalitions=Array([[False, False, False],
       [ True,  True,  True],
       [False, False,  True],
       [ True,  True, False],
       [ True, False, False],
       [False,  True,  True],
       [False,  True, False],
       [ T